# Homework 4 — First Agentic Workflow

## PSYC 1111 Health Psychology Course Assistant

This notebook extends the existing Health Psychology RAG project with a simple controlled agentic workflow.

The workflow:

1. receives a user question;
2. selects a deterministic route;
3. calls a retrieval tool;
4. observes the retrieved context;
5. generates a grounded answer;
6. updates workflow state;
7. returns the final answer.

The implementation also compares a weak prompt with a grounded RAG prompt to demonstrate how prompt design affects model behavior.

## Stage 1 — Connect the repository and prepare the environment

In [1]:
!pip install -q --no-cache-dir \
    google-genai \
    numpy==1.26.4 \
    scipy==1.13.1 \
    scikit-learn==1.5.1 \
    pandas==2.2.2 \
    sentence-transformers==3.0.1 \
    faiss-cpu==1.8.0.post1

In [2]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess


GITHUB_USER = "swanksenia"
REPO_NAME = "health-psychology-rag-kb"

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content") / REPO_NAME


github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )


askpass_path = Path("/content/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass_path.chmod(0o700)


git_environment = os.environ.copy()
git_environment["GITHUB_TOKEN"] = github_token
git_environment["GIT_ASKPASS"] = str(askpass_path)
git_environment["GIT_TERMINAL_PROMPT"] = "0"


if (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull"],
        check=True,
        env=git_environment,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_ROOT)],
        check=True,
        env=git_environment,
    )


print("Project root:", PROJECT_ROOT)

Project root: /content/health-psychology-rag-kb


## Stage 2 - OpenAI API key

In [18]:
from google import genai


GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found in Colab Secrets."
    )


GEMINI_MODEL = "gemini-3.5-flash"

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Model:", GEMINI_MODEL)

Model: gemini-3.5-flash


## Step 4. Завантажуємо вже існуючий retrieval

Не перебудовуємо FAISS, вже є:

data/processed/chunks_for_retrieval.jsonl

index/faiss.index

In [4]:
import json
import re

import faiss
from sentence_transformers import SentenceTransformer


CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_for_retrieval.jsonl"
)

INDEX_PATH = (
    PROJECT_ROOT
    / "index"
    / "faiss.index"
)

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            records.append(json.loads(line))

    return records


chunks = load_jsonl(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


print("Chunks loaded:", len(chunks))
print("FAISS vectors:", index.ntotal)

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Chunks loaded: 474
FAISS vectors: 474


## Step 5. Metadata

Беремо майже буквально з попереднього retrieval_improved.py.

In [5]:
DOCUMENT_TYPE_BY_DOCUMENT = {
    "health_psychology_course_syllabus": "syllabus",
    "ogden_2019_health_psychology": "textbook",
    "michie_2011_behaviour_change_wheel": "research_article",
    "wright_2019_3p_disease_model": "research_article",
}


def add_metadata(chunks):
    for chunk in chunks:
        document_id = chunk["document_id"]

        chunk["metadata"] = {
            "document_id": document_id,
            "source_file": chunk.get("source_file"),
            "section": chunk.get("section"),
            "chunk_index": chunk.get("chunk_index"),
            "document_type": DOCUMENT_TYPE_BY_DOCUMENT[
                document_id
            ],
        }


add_metadata(chunks)

## Step 6. Tool №1 — content retrieval

In [6]:
def semantic_search(
    query,
    model,
    index,
    chunks,
    top_k=3,
):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
    )

    query_embedding = query_embedding.astype(
        "float32"
    )

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        top_k,
    )

    results = []

    for score, chunk_index in zip(
        scores[0],
        indices[0],
    ):
        if chunk_index == -1:
            continue

        chunk = chunks[chunk_index]

        results.append(
            {
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "document_id": chunk["document_id"],
                "section": chunk.get("section"),
                "text": chunk["text"],
                "metadata": chunk["metadata"],
            }
        )

    return results

### А tool:

In [7]:
def retrieve_course_content(
    question,
):
    return semantic_search(
        question,
        embedding_model,
        index,
        chunks,
        top_k=3,
    )

## Step 7. Tool №2 — course structure retrieval

Щоб route справді був інший, другий tool шукає тільки syllabus.

In [8]:
def retrieve_course_structure(
    question,
):
    candidates = semantic_search(
        question,
        embedding_model,
        index,
        chunks,
        top_k=10,
    )

    syllabus_results = [
        result
        for result in candidates
        if result["metadata"]["document_type"]
        == "syllabus"
    ]

    return syllabus_results[:3]

## Step 8. Router

In [9]:
COURSE_STRUCTURE_KEYWORDS = [
    "unit",
    "course",
    "syllabus",
    "read",
    "reading",
    "topic",
    "topics",
    "learning objective",
]


HEALTH_PSYCHOLOGY_KEYWORDS = [
    "health",
    "psychology",
    "biopsychosocial",
    "stress",
    "pain",
    "behaviour",
    "behavior",
    "com-b",
    "3p",
    "illness",
    "intervention",
]


def route_question(question):
    q = question.lower()

    if any(
        keyword in q
        for keyword in COURSE_STRUCTURE_KEYWORDS
    ):
        return "course_structure"

    if any(
        keyword in q
        for keyword in HEALTH_PSYCHOLOGY_KEYWORDS
    ):
        return "course_content"

    return "clarification"

## Step 9. Formatting retrieved context

In [10]:
def format_context(results):
    blocks = []

    for result in results:
        blocks.append(
            f"""
Source chunk ID: {result['chunk_id']}
Source document: {result['document_id']}
Section: {result['section']}
Content:
{result['text']}
""".strip()
        )

    return "\n\n---\n\n".join(blocks)

## Step 10. Weak prompt

In [11]:
def build_weak_prompt(
    question,
    context,
):
    return f"""
Answer the user question using the context.

Context:
{context}

Question:
{question}
""".strip()

## Step 11. Grounded prompt

In [12]:
def build_grounded_prompt(
    question,
    context,
):
    return f"""
You are a Health Psychology course assistant.

Your task:
Answer the user question using only the provided context.

Rules:
- Do not use external knowledge.
- Do not invent missing information.
- If the provided context does not contain enough information, say:
  "I do not have enough information in the provided course materials."
- Mention the source chunk ID(s) used in the answer.
- Keep the answer concise and clear.

Context:
{context}

User question:
{question}

Answer:
""".strip()

## Step 12. LLM call через Responses API

In [13]:
def call_model(prompt):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )

    return response.text

## Step 13. State (agentic part)

In [14]:
def create_state(question):
    return {
        "user_goal": question,
        "selected_route": None,
        "tool_calls": [],
        "observations": [],
        "final_answer": None,
    }

## Step 14. Agent flow

In [15]:
def run_agent(
    question,
    prompt_mode="grounded",
):
    state = create_state(question)

    # Step 1: route
    route = route_question(question)
    state["selected_route"] = route

    # Step 2: execute route
    if route == "course_content":

        results = retrieve_course_content(
            question
        )

        state["tool_calls"].append(
            {
                "tool": "retrieve_course_content",
                "input": question,
            }
        )

    elif route == "course_structure":

        results = retrieve_course_structure(
            question
        )

        state["tool_calls"].append(
            {
                "tool": "retrieve_course_structure",
                "input": question,
            }
        )

    else:
        state["observations"].append(
            "The question could not be mapped "
            "to the supported Health Psychology workflows."
        )

        state["final_answer"] = (
            "Could you clarify your question? "
            "I can help with Health Psychology "
            "course content or course structure."
        )

        return state

    # Step 3: observation
    context = format_context(results)

    state["observations"].append(
        {
            "retrieved_chunks": [
                result["chunk_id"]
                for result in results
            ],
            "context": context,
        }
    )

    # Step 4: prompt
    if prompt_mode == "weak":
        prompt = build_weak_prompt(
            question,
            context,
        )

    else:
        prompt = build_grounded_prompt(
            question,
            context,
        )

    # Step 5: model action
    answer = call_model(prompt)

    state["tool_calls"].append(
        {
            "tool": "gemini_generate_content",
            "model": GEMINI_MODEL,
            "prompt_mode": prompt_mode,
        }
    )

    # Step 6: final state update
    state["final_answer"] = answer

    return state

## Step 15. Test one

In [19]:
test_state = run_agent(
    "What is the biopsychosocial model of health?",
    prompt_mode="grounded",
)

test_state

{'user_goal': 'What is the biopsychosocial model of health?',
 'selected_route': 'course_content',
 'tool_calls': [{'tool': 'retrieve_course_content',
   'input': 'What is the biopsychosocial model of health?'},
  {'tool': 'gemini_generate_content',
   'model': 'gemini-3.5-flash',
   'prompt_mode': 'grounded'}],
 'observations': [{'retrieved_chunks': ['ogden_2019_health_psychology__0013',
    'wright_2019_3p_disease_model__0010',
    'wright_2019_3p_disease_model__0007'],
   'context': 'Source chunk ID: ogden_2019_health_psychology__0013\nSource document: ogden_2019_health_psychology\nSection: 1.The Biopsychosocial Model\nContent:\nsmoking), pressures to change behavior (e.g. peer group expectations, parental pressure), social values on health (e.g. whether health was regarded as a good or a bad thing), social class, the environment, and ethnicity. #### Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980) ![Fig 1 The biopsychosocial model of health and illness

### Observation 1

The agent correctly routed the question to the `course_content` workflow, retrieved three relevant chunks, and generated a grounded answer using only the retrieved context.

The final answer included source chunk IDs, demonstrating that the citation requirement in the grounded prompt was followed.

## Step 15: Test two (weak vs grounded)



In [20]:
comparison_question = (
    "What is the biopsychosocial model of health?"
)

weak_result = run_agent(
    comparison_question,
    prompt_mode="weak",
)

grounded_result = run_agent(
    comparison_question,
    prompt_mode="grounded",
)

print("=== WEAK PROMPT ===")
print(weak_result["final_answer"])

print("\n=== GROUNDED PROMPT ===")
print(grounded_result["final_answer"])

=== WEAK PROMPT ===
Based on the provided context, the biopsychosocial model of health:

* **Explains health and illness** through the interaction of biological, psychological, and social (or socio-environmental) factors.
* **Was developed as a response**, in part, to biological reductionism.
* **States that multiple disciplines contribute to health and disease**, playing an integral role across various fields, including health psychology and behavioral medicine.
* **Lacks a framework** (as noted by some critics) for understanding *how* these biological, psychological, and socio-environmental factors contribute to each stage of disease development, maintenance, or treatment.

=== GROUNDED PROMPT ===
Based on the provided course materials, the biopsychosocial model is a model that:

* Explains health and illness through the interaction of biological, psychological, and social (or socio-environmental) factors (Source: ogden_2019_health_psychology__0013, wright_2019_3p_disease_model__0010

### Observation 2

Both prompts produced a correct answer because the retrieved context was relevant and sufficient.

However, the weak prompt did not provide explicit source references, while the grounded prompt cited the exact chunk IDs used to support the answer.

This shows that a weak prompt may work in a simple case, but it does not enforce traceability. The grounded prompt provides more controlled and reproducible behavior by explicitly defining the context boundary and citation requirement.

## Step 15: Test three (fallback)



In [21]:
fallback_question = (
    "In health psychology, what are the DSM-5 "
    "diagnostic criteria for schizophrenia?"
)

weak_fallback_result = run_agent(
    fallback_question,
    prompt_mode="weak",
)

grounded_fallback_result = run_agent(
    fallback_question,
    prompt_mode="grounded",
)

print("=== WEAK PROMPT ===")
print(weak_fallback_result["final_answer"])

print("\n=== GROUNDED PROMPT WITH FALLBACK ===")
print(grounded_fallback_result["final_answer"])

=== WEAK PROMPT ===
Based on the provided context, there is no mention of the DSM-5 diagnostic criteria for schizophrenia.

=== GROUNDED PROMPT WITH FALLBACK ===
I do not have enough information in the provided course materials.


In [22]:
print("=== RETRIEVED CHUNKS ===")

for chunk_id in grounded_fallback_result["observations"][0]["retrieved_chunks"]:
    print("-", chunk_id)

=== RETRIEVED CHUNKS ===
- wright_2019_3p_disease_model__0008
- ogden_2019_health_psychology__0005
- wright_2019_3p_disease_model__0007


### Observation 3

The retrieved chunks were related to health psychology but did not contain the DSM-5 diagnostic criteria for schizophrenia.

The weak prompt also avoided inventing an answer in this run, but it produced its own wording for the missing-information case.

The grounded prompt followed the explicitly defined fallback rule and returned the expected response:

> "I do not have enough information in the provided course materials."

This demonstrates that an explicit fallback instruction makes model behavior more predictable and consistent when retrieval does not provide enough evidence.

## Step 15. Test four (course structure route)

This test checks whether the deterministic router correctly identifies a course-navigation question and sends it to the `course_structure` workflow.

Expected flow:

user question → course_structure route → retrieve_course_structure tool → syllabus chunks → grounded answer

In [23]:
structure_question = (
    "What topics are covered in the Health Psychology course?"
)

structure_result = run_agent(
    structure_question,
    prompt_mode="grounded",
)

print("=== ROUTE ===")
print(structure_result["selected_route"])

print("\n=== TOOLS CALLED ===")
for tool_call in structure_result["tool_calls"]:
    print("-", tool_call["tool"])

print("\n=== RETRIEVED CHUNKS ===")
for chunk_id in structure_result["observations"][0]["retrieved_chunks"]:
    print("-", chunk_id)

print("\n=== FINAL ANSWER ===")
print(structure_result["final_answer"])

=== ROUTE ===
course_structure

=== TOOLS CALLED ===
- retrieve_course_structure
- gemini_generate_content

=== RETRIEVED CHUNKS ===
- health_psychology_course_syllabus__0000
- health_psychology_course_syllabus__0006
- health_psychology_course_syllabus__0003

=== FINAL ANSWER ===
Based on the provided course materials, the course covers:

* The dynamic interaction between biological, social, and psychological factors that influence physical health and illness, with the goal of promoting overall well-being and preventing diseases (Source: `health_psychology_course_syllabus__0000`).
* Theoretical frameworks and concepts that form the foundation of a psychological perspective on physical health (Source: `health_psychology_course_syllabus__0003`).


### Observation 4

The router correctly identified the question as a `course_structure` request.

The `retrieve_course_structure` tool returned only syllabus chunks, which confirms that this route uses a different retrieval strategy from the general `course_content` workflow.

The grounded answer was generated only from the retrieved syllabus context and included source chunk IDs.

## Step 15: Test five (clarification route)

How the workflow handles a question that cannot be mapped to the supported Health Psychology routes.

Expected flow:

user question → clarification route → no retrieval tool → clarification response


In [24]:
clarification_question = (
    "Tell me something interesting."
)

clarification_result = run_agent(
    clarification_question,
    prompt_mode="grounded",
)

print("=== ROUTE ===")
print(clarification_result["selected_route"])

print("\n=== TOOLS CALLED ===")
print(clarification_result["tool_calls"])

print("\n=== OBSERVATIONS ===")
print(clarification_result["observations"])

print("\n=== FINAL ANSWER ===")
print(clarification_result["final_answer"])

=== ROUTE ===
clarification

=== TOOLS CALLED ===
[]

=== OBSERVATIONS ===
['The question could not be mapped to the supported Health Psychology workflows.']

=== FINAL ANSWER ===
Could you clarify your question? I can help with Health Psychology course content or course structure.


### Observation 5

The router correctly sent the unsupported question to the `clarification` route.

No retrieval or model tool was called. Instead, the workflow returned a controlled clarification message and preserved the decision in the workflow state.

This demonstrates that the agent does not need to call a tool for every user request.